<a href="https://colab.research.google.com/github/sureshaiexec/AWS-codedeploy/blob/feature%2Fsampletest/RAG-Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers sentence-transformers faiss-cpu langchain

In [ ]:
import os
from langchain_community.text_splitter import RecursiveCharacterTextSplitter

# Load our document
with open("/content/sample_data/my_knowledge.txt") as f:
    knowledge_text = f.read()

# 1. Initialize the Text Splitter
# This splitter is smart. It tries to split on paragraphs ("\n\n"),
# then newlines ("\n"), then spaces (" "), to keep semantically
# related text together as much as possible.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,  # Max size of a chunk
    chunk_overlap=20, # Overlap to maintain context between chunks
    length_function=len
)

# 2. Create the chunks
chunks = text_splitter.split_text(knowledge_text)

print(f"We have {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---\n{chunk}\n")

In [ ]:
!pip install langchain_community

In [ ]:
!pip install langchain_text_splitters

In [ ]:
!ls /content/sample_data/

The `my_knowledge.txt` file is located in `/content/sample_data/`. I will update the code to use the correct path.

In [ ]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load our document
with open("/content/sample_data/my_knowledge.txt") as f:
    knowledge_text = f.read()

# 1. Initialize the Text Splitter
# This splitter is smart. It tries to split on paragraphs ("\n\n"),
# then newlines ("\n"), then spaces (" "), to keep semantically
# related text together as much as possible.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,  # Max size of a chunk
    chunk_overlap=20, # Overlap to maintain context between chunks
    length_function=len
)

# 2. Create the chunks
chunks = text_splitter.split_text(knowledge_text)

print(f"We have {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---\n{chunk}\n")

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load the embedding model
# 'all-MiniLM-L6-v2' is a fantastic, fast, and small model.
# It runs 100% on your local machine.
embedding_model = SentenceTransformer('all-MiniLM-L6-v2') # Renamed from 'model'

# 2. Embed all our chunks
# This will take a moment as it "reads" and "understands" each chunk.
chunk_embeddings = embedding_model.encode(chunks) # Use embedding_model

print(f"Shape of our embeddings: {chunk_embeddings.shape}")

In [ ]:
import faiss
import numpy as np

# Get the dimension of our vectors (e.g., 384)
d = chunk_embeddings.shape[1]

# 1. Create a FAISS index
# IndexFlatL2 is the simplest, most basic index. It calculates
# the exact distance (L2 distance) between our query and all vectors.
index = faiss.IndexFlatL2(d)

# 2. Add our chunk embeddings to the index
# We must convert to float32 for FAISS
index.add(np.array(chunk_embeddings).astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load the T5 model and tokenizer directly
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-small')
text_generation_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-small')

# Create a custom generator function for sequence-to-sequence models
def custom_generator(prompt_text, max_new_tokens=100):
    inputs = tokenizer(prompt_text, return_tensors='pt')
    outputs = text_generation_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- This is our RAG pipeline function ---
def answer_question(query):
    # 1. RETRIEVE
    # Embed the user's query
    # Use 'embedding_model' here, which was loaded in cell d9xvpMEOsc3r
    query_embedding = embedding_model.encode([query]).astype('float32')

    # Search the FAISS index for the top k (e.g., k=2) most similar chunks
    k = 2
    distances, indices = index.search(query_embedding, k)

    # Print the distance for debugging and tuning the threshold
    print(f"DEBUG: Distance to most relevant chunk: {distances[0][0]:.4f}")

    # Add a relevance check based on the distance of the most similar chunk
    # Smaller L2 distance means higher similarity. Higher values indicate less similarity.
    # The 'relevance_threshold' needs to be adjusted based on the specific embedding model and data.
    relevance_threshold = 1.0 # Adjusted threshold based on observed distances

    if distances[0][0] > relevance_threshold:
        print("--- CONTEXT ---\nNo sufficiently relevant context found.")
        return "I don't know."

    # Get the actual text chunks from our original 'chunks' list
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # 2. AUGMENT
    # This is the "magic prompt." We combine the retrieved context
    # with the user's query.
    prompt_template = f"Context: {context}\nQuestion: {query}\nAnswer:"

    # 3. GENERATE
    # Feed the augmented prompt to our custom generative model
    answer = custom_generator(prompt_template, max_new_tokens=100)
    print(f"--- CONTEXT ---\n{context}\n")
    return answer

In [ ]:
query_1 = "What is the WFH policy?"
print(f"Query: {query_1}")
print(f"Answer: {answer_question(query_1)}\n")

In [ ]:
query_2 = "What is the company's dental plan?"
print(f"Query: {query_2}")
print(f"Answer: {answer_question(query_2)}\n")